# Part 5: DeepLabV3+ Land Cover Segmentation

理解空洞卷积、ASPP、多尺度上下文与轻量 decoder，面向多类别土地覆盖分割。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`backbone → low/high-level features → ASPP(rates) → upsample → decoder → land-cover logits`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **Atrous convolution**：在不继续下采样的前提下扩大感受野。
- **ASPP**：并行使用多个 dilation rate 和全局池化捕捉多尺度上下文。
- **Output stride**：控制特征图分辨率与计算量，常见 8 或 16。
- **Decoder**：融合浅层边界特征，改善道路、岸线和地块边缘。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part5_deeplabv3plus'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python scripts/real_training/train_segmentation.py --model deeplabv3plus --data-root data/landcoverai --output outputs/part5_deeplabv3plus --epochs 1 --train-limit 32 --val-limit 16'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

为 10 m 分辨率、目标尺度约 30–300 m 的影像提出一组 ASPP rates，并说明 output stride 的影响。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：dilation 过大出现 gridding；忽略类别不平衡；上采样 align_corners 设置不一致；训练/验证地块重叠。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例